# Introducción a la visión artificial: De la Detección a la Acción

## Descripción General
Este cuaderno implementa un sistema avanzado donde la Inteligencia Artificial actúa como el "cerebro" central. Utilizaremos YOLO (You Only Look Once) para que el sistema logre identificar, clasificar y tomar decisiones en tiempo real sobre los objetos presentes en su entorno (ej. diferenciar entre botellas y tazas/latas).

## Objetivos del Curso
1. **Despliegue de CNNs:** Implementar YOLO (versión Nano/Tiny) en entornos locales para detección múltiple.
2. **Toma de decisiones:** Desarrollar algoritmos autónomos basados en etiquetas de clasificación.
3. **Optimización:** Ajustar parámetros computacionales para garantizar el procesamiento en tiempo real.
4. **Lazo cerrado (IA - Hardware):** Integrar el sistema de visión con un microcontrolador vía comunicación serial (Bluetooth/USB).

In [ ]:
# ==============================================================================
# INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================
# (Descomenta la siguiente línea si necesitas instalar las librerías en tu entorno)
# !pip install ultralytics opencv-python pyserial

import cv2
import time
import serial
from ultralytics import YOLO

print("Librerías importadas correctamente. Entorno listo para visión artificial.")

## 1. Redes Neuronales: Implementación de YOLO
Para lograr procesamiento en tiempo real, utilizaremos `yolov8n.pt` (la versión más compacta y rápida de la arquitectura YOLO, equivalente moderno al clásico YOLO Tiny). Este modelo nos permite mantener un alto frame-rate (FPS) sin saturar los recursos del CPU/GPU.

In [ ]:
# Cargar el modelo YOLO. Si es la primera vez, se descargará automáticamente (aprox. 6 MB).
# Este modelo está preentrenado con el dataset COCO (80 clases de objetos comunes).
modelo = YOLO("yolov8n.pt") 

print("Arquitectura YOLO cargada con éxito.")

## 2. Programación del Clasificador y Diccionario de Decisiones
La IA por sí sola solo detecta; necesitamos estructurar la lógica para que tome decisiones. Crearemos un **diccionario de decisiones** que mapee la etiqueta (clase) identificada con un comando específico en bytes, el cual será enviado al microcontrolador.

In [ ]:
# Creación del diccionario para la toma de decisiones.
# Mapeamos los objetos detectados a comandos lógicos para los servomotores.
# Por ejemplo, separaremos 'bottle' (botellas) y 'cup' (vasos/latas).

diccionario_decisiones = {
    'bottle': b'A',  # Comando 'A': Mover servomotor 1 (ej. contenedor de plástico)
    'cup': b'B'      # Comando 'B': Mover servomotor 2 (ej. contenedor de aluminio/vidrio)
}

print("Diccionario de decisiones estructurado:")
for objeto, comando in diccionario_decisiones.items():
    print(f" - Si detecta '{objeto}', enviará la señal: {comando}")

## 3. Conexión Electrónica y Sincronización
Para lograr el **sistema de lazo cerrado**, estableceremos comunicación Serial. Esto permite interactuar mediante módulos Bluetooth (HC-05/HC-06) o conexión directa por USB al microcontrolador.

**Habilidad clave:** Es vital sincronizar el tiempo de procesamiento computacional con el tiempo mecánico. Un servomotor requiere tiempo físico para moverse; si enviamos comandos a la velocidad de la inferencia (ej. 30 comandos por segundo), el hardware se saturará.

In [ ]:
# Configuración del puerto de comunicación vía Bluetooth/USB.
# Ajusta el puerto según el sistema operativo:
# - Linux: '/dev/rfcomm0' (Bluetooth) o '/dev/ttyUSB0' (Cable)
# - Windows: 'COM3', 'COM4', etc.

PUERTO = '/dev/rfcomm0' 
BAUD_RATE = 9600

try:
    microcontrolador = serial.Serial(PUERTO, BAUD_RATE, timeout=1)
    time.sleep(2) # Pausa necesaria para estabilizar la conexión serial
    print(f"Conexión exitosa con el hardware en {PUERTO}.")
except serial.SerialException:
    print(f"ADVERTENCIA: No se pudo abrir {PUERTO}. Ejecutando en modo de simulación sin hardware físico.")
    microcontrolador = None

## 4. Despliegue del Clasificador en Tiempo Real
A continuación, manipularemos el flujo de video usando OpenCV e integraremos todo el pipeline. 
Implementaremos **filtrado de detecciones** mediante un umbral de confianza (*Confidence Score*) para evitar falsos positivos y estableceremos el **control lógico del tiempo** (Time Sincronization) para coordinar las reacciones físicas.

In [ ]:
# Inicializar la cámara web (0 es la cámara principal)
cap = cv2.VideoCapture(0)

# ==========================================================
# PARÁMETROS DE OPTIMIZACIÓN Y SINCRONIZACIÓN
# ==========================================================
UMBRAL_CONFIANZA = 0.65       # Filtrado: Solo acepta detecciones con más de 65% de seguridad (evita falsos positivos)
TIEMPO_REACCION_FISICA = 2.0  # Segundos que tarda el sistema físico (servomotor) en ejecutar la acción
ultimo_comando = 0            # Registro temporal del último comando enviado

print("Iniciando despliegue de video. Presiona 'q' en la ventana para salir.")

while cap.isOpened():
    exito, frame = cap.read()
    if not exito:
        print("No se pudo recibir el flujo de video.")
        break

    # 1. Inferencia Optimizada: imgsz reduce la resolución interna para agilizar el tiempo real
    resultados = modelo.predict(source=frame, imgsz=320, conf=UMBRAL_CONFIANZA, verbose=False)
    
    # 2. Análisis de las detecciones del frame actual
    for resultado in resultados:
        for caja in resultado.boxes:
            # Extraer clase, confianza y coordenadas
            clase_id = int(caja.cls[0])
            confianza = float(caja.conf[0])
            nombre_clase = modelo.names[clase_id]
            
            # Dibujar Bounding Box y etiquetas mediante OpenCV
            x1, y1, x2, y2 = map(int, caja.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            etiqueta = f"{nombre_clase}: {confianza:.2f}"
            cv2.putText(frame, etiqueta, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            # 3. Consulta al diccionario de decisiones
            if nombre_clase in diccionario_decisiones:
                comando = diccionario_decisiones[nombre_clase]
                
                # 4. Sincronización Computadora-Física (Evitar saturación del buffer serial)
                tiempo_actual = time.time()
                if (tiempo_actual - ultimo_comando) > TIEMPO_REACCION_FISICA:
                    if microcontrolador:
                        microcontrolador.write(comando)
                    
                    print(f"✅ Decisión autónoma: {nombre_clase} detectado. Enviando orden {comando} al hardware.")
                    ultimo_comando = tiempo_actual # Actualizamos el reloj

    # Mostrar la interfaz visual
    cv2.imshow("Sistema de Vision Artificial Lazo Cerrado", frame)

    # Condición de salida (tecla 'q')
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Limpieza y cierre seguro de recursos
cap.release()
cv2.destroyAllWindows()
if microcontrolador:
    microcontrolador.close()
    print("Conexión serial terminada de forma segura.")